### 6-بهتره برای فروش بیشتر چه دسته های تلاش کنیم؟ و فروش  در چه دسته هایی احتمالا باعث کاهش ارزش برند ما میشه؟
### 7-میخواهیم یک سیستم امتیاز دهی به فروشنده ها داشته باشیم؟
### 8-آیا زمان اماده سازی کالا با فروشش ارتباطی داره؟
### 9- میخواهیم فروشنده ها را براساس معیارهایی مثل تنوع محصولات تعداد و حجم فروش  و معیارهای دیگه به چند دسته تقسیم کنیم

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
labeled_comments = pd.read_csv(f"{data_path}/all_labeled_comments.csv")
map_category = pd.read_csv(f"{data_path}/category_map.csv")
category_mapping = dict(zip(map_category['sub_category'], map_category['main_category']))
products['main_category'] = products['categoryTitle'].map(category_mapping).fillna("سایر")
products[['categoryTitle','main_category']].sample()

### 6-بهتره برای فروش بیشتر چه دسته های تلاش کنیم؟ و فروش  در چه دسته هایی احتمالا باعث کاهش ارزش برند ما میشه؟

محصولاتی که نارضایتی زیادی دارن باعث کاهش ارزش برند میشه

In [ ]:
products['rating_average'].value_counts().sort_index()

In [ ]:
products['rating_count'].value_counts().sort_index()

In [ ]:
1634181 / products.shape[0]

حدود ۶۸ درصد محصولات اصلا امتیاز ندارن
چون تعداد امتیاز ها برای هر محصول متفاوته یه میانگین وزنی می گیریم تا تعداد امتیاز ها رو در روند ارزیابی مون دخیل کنیم

In [ ]:
products['weighted_rating'] = (products['rating_average'] * products['rating_count']) /(products['rating_average'] + products['rating_count'])
products[['rating_average','rating_count', 'weighted_rating']]

In [ ]:
data = products[products['rating_count']!=0].groupby('main_category')['weighted_rating'].agg(
    ['mean', 'count']).reset_index().sort_values(by='mean', ascending=False)

fig = px.bar(data, x='main_category', y='mean')
fig.show()

In [ ]:
data = products[products['rating_count']!=0].groupby('main_category')['rating_average'].agg(
    ['mean', 'count']).reset_index().sort_values(by='mean', ascending=False)

fig = px.bar(data, x='main_category', y='mean')
fig.show()

In [ ]:
comments = pd.merge(labeled_comments, reviews[['_id', 'productId']], left_on='_id', right_on='_id')
comments['sentiment_by_parsbert'] = comments['sentiment_by_parsbert'].map({
    'Positive': 1,
    'Negative': 0
})
comments

In [ ]:
data = comments.groupby('productId')['sentiment_by_parsbert'].agg(['mean','count']).reset_index()
data = pd.merge(data, products[['_id', 'main_category']], left_on='productId', right_on='_id')
data = data.groupby('main_category')['mean'].agg(['mean', 'count']).reset_index().sort_values(by='mean', ascending=False)
data

In [ ]:
fig = px.bar(data, x='main_category', y='mean')
fig.update_layout(yaxis=dict(range=[0.8, 1]))
fig.show()

از تعداد ستاره ها هم می تونیم استفاده کنیم

### 7-میخواهیم یک سیستم امتیازدهی به فروشنده ها داشته باشیم؟

x = تعداد فروش - ارسال رایگان(چند درصد محصولاتش) - زمان اماده سازی - احساسات کامنت هاش(اگه داره) - میانگین ستاره هاش(اگه داره)  - 
<br>
y = میانگین امتیاز 


In [ ]:
products['isFreeShipping'] = products['isFreeShipping'].fillna(False).astype('int32')
products['has_delivery'] = products['has_delivery'].fillna(False).astype('int32')

In [ ]:
vendor_commnets = pd.merge(comments, products[['_id', 'vendor_id']], left_on='productId', right_on='_id')
vendor_commnets = pd.merge(vendor_commnets, reviews[['_id', 'star']], left_on='_id_x', right_on='_id')

In [ ]:
# check medain
train_data = products.groupby('vendor_id')['sales_count_week'].sum().reset_index()
train_data['rating'] = products.groupby('vendor_id')['rating_average'].mean().reset_index()['rating_average']
train_data['isFreeShipping'] = products.groupby('vendor_id')['isFreeShipping'].mean().reset_index()['isFreeShipping']
train_data['preparationDays'] = products.groupby('vendor_id')['preparationDays'].mean().reset_index()['preparationDays']
train_data['has_delivery'] = products.groupby('vendor_id')['has_delivery'].mean().reset_index()['has_delivery']
train_data['has_comment'] = train_data['vendor_id'].isin(vendor_commnets['vendor_id']).astype(int)
train_data['comment_sentiment'] = 0.0
train_data.loc[train_data['has_comment']==1,'comment_sentiment'] = vendor_commnets.groupby('vendor_id')['sentiment_by_parsbert'].mean().values
train_data['star_avg'] = 0.0
train_data.loc[train_data['has_comment']==1,'star_avg'] = vendor_commnets.groupby('vendor_id')['star'].mean().values
vendor_ids = train_data['vendor_id']
train_data.drop('vendor_id', axis=1, inplace=True)

In [ ]:
train_data = train_data[train_data['rating']!=0]
train_data

In [ ]:
train_data[train_data['sales_count_week']!=0].hist(figsize=(10, 6), bins=30, edgecolor='black')
plt.tight_layout()
plt.show()


In [ ]:
X = train_data.drop(columns=["rating"])  
y = train_data["rating"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    "None": None,  
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Support Vector Regression": SVR(kernel="rbf", C=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42),
    "CatBoost": CatBoostRegressor(n_estimators=100, verbose=0, random_state=42),
}

results = []
for scaler_name, scaler in scalers.items():
    for model_name, model in models.items():
        if scaler_name != "None" and isinstance(model, (RandomForestRegressor, XGBRegressor, LGBMRegressor, CatBoostRegressor)):
            continue
        
        steps = []
        if scaler:
            steps.append(("scaler", scaler))
        steps.append(("model", model))
        pipeline = Pipeline(steps)

        scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
        mean_score = np.mean(   )

        results.append((scaler_name, model_name, mean_score))
        print(f"{scaler_name} + {model_name}: R² = {mean_score:.4f}")

best_model = max(results, key=lambda x: x[2])
print("\n🎉 Best Model:", best_model[1])
print("📊 Best Scaler:", best_model[0])
print(f"💯 Best R² Score: {best_model[2]:.4f}")

In [ ]:
from sklearn.metrics import r2_score
Mlmodel = LGBMRegressor(n_estimators=100, random_state=42)
Mlmodel.fit(X_train, y_train)
y_pred = Mlmodel.predict(X_test)
print(f"r2:{r2_score(y_test, y_pred)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

class RegressionNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  
        )

    def forward(self, x):
        return self.model(x)

class R2Loss(nn.Module):
    def forward(self, y_pred, y_true):
        ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
        ss_residual = torch.sum((y_true - y_pred) ** 2)
        return 1 - (ss_residual / ss_total) 

input_size = X_train.shape[1]
model = RegressionNN(input_size)
criterion = R2Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train)
    loss = -criterion(y_pred, y_train) 
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    y_pred_test = model(X_test)
    r2 = r2_score(y_test.numpy(), y_pred_test.numpy())
    print(f"\n🎯 Final R² Score on Test Data: {r2:.4f}")


In [ ]:
id = 13024
sample = np.array([X.iloc[id].values])  
sample_scaled = scaler.transform(sample)  
sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32) 
model.eval()  

with torch.no_grad():  
    prediction = model(sample_tensor)

print("Predicted value DL:", prediction.item())  #
print("Predicted value ML:", Mlmodel.predict(sample)) 
print("real value :", y.iloc[id])  

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(x=train_data['star_avg'], y=train_data['rating'], alpha=0.6, edgecolors="k")
plt.grid(True)
plt.show()

### 8-آیا زمان اماده سازی کالا با فروشش ارتباطی داره؟


In [ ]:
products[products['sales_count_week']>1][['sales_count_week', 'preparationDays']].corr()

### 9- میخواهیم فروشنده ها را براساس معیارهایی مثل تنوع محصولات تعداد و حجم فروش  و معیارهای دیگه به چند دسته تقسیم کنیم

In [ ]:
data = products[(products['sales_count_week']>0) & (products['rating_count']>0)].groupby('vendor_id')['sales_count_week'].sum().reset_index()
data['num_product'] = products[(products['sales_count_week']>0) & (products['rating_count']>0)].groupby('vendor_id')['_id'].size().values
data['rating'] = products[(products['sales_count_week']>0) & (products['rating_count']>0)].groupby('vendor_id')['rating_average'].mean().values

In [ ]:
fig = px.scatter_3d(data , x='sales_count_week', y='num_product', z='rating',
                    hover_data=['vendor_id'])
fig.show()